In [20]:
#New RAG Model
# radar_rag_engine.py
# ---------------------------------------------------------------
# Offline Radar Telemetry Interpreter
# Requires: Ollama running locally with Qwen model downloaded
# ---------------------------------------------------------------

import json
import time
from typing import List, Dict, Any, Optional
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
import requests
import pandas as pd

# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5-coder:7b-instruct-q5_K_M"

registry = get_registry().get("sentence-transformers")
embedding_model = registry.create(name="BAAI/bge-small-en-v1.5")


def call_local_qwen(prompt: str, json_mode: bool = False) -> str:
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    if json_mode:
        payload["format"] = "json"

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json().get("response", "").strip()
        else:
            return f"Error: Ollama returned status {response.status_code}"
    except requests.exceptions.Timeout:
        return "Error: Ollama request timed out (120s)"
    except requests.exceptions.ConnectionError:
        return "Error: Cannot connect to Ollama at http://localhost:11434"
    except Exception as e:
        return f"Error connecting to local LLM: {str(e)}"


class RadarLogSchema(LanceModel):
    id: str
    timestamp: str
    radar_id: str
    azimuth: float
    elevation: float
    voltage_mv: int
    error_code: str
    log_message: str
    vector: Vector(384) = embedding_model.VectorField()


try:
    db = lancedb.connect("./local_radar_db")
    table_name = "radar_telemetry"

    if table_name in db.table_names():
        table = db.open_table(table_name)
    else:
        table = db.create_table(table_name, schema=RadarLogSchema)
except Exception as e:
    print(f"[ERROR] Failed to connect to LanceDB: {str(e)}")
    raise


class ContinuumMemory:
    def __init__(self):
        self.state_file = "radar_continuum_memory.json"
        self.initialize_memory()

    def initialize_memory(self):
        try:
            with open(self.state_file, 'r') as f:
                self.memory = json.load(f)
        except FileNotFoundError:
            self.memory = {
                "last_processed_timestamp": "1970-01-01 00:00:00",
                "known_critical_faults": [],
                "active_anomalies": {}
            }
            self.save()
        except json.JSONDecodeError:
            print(f"[WARNING] Memory file corrupted, starting fresh")
            self.memory = {
                "last_processed_timestamp": "1970-01-01 00:00:00",
                "known_critical_faults": [],
                "active_anomalies": {}
            }
            self.save()

    def save(self):
        try:
            with open(self.state_file, 'w') as f:
                json.dump(self.memory, f, indent=2)
        except Exception as e:
            print(f"[WARNING] Failed to save memory: {str(e)}")

    def update_state(self, new_fault: str, radar_id: str):
        if new_fault not in self.memory["known_critical_faults"]:
            self.memory["known_critical_faults"].append(new_fault)
        self.memory["active_anomalies"][radar_id] = f"Last fault tracked: {new_fault}"
        self.save()

    def get_context_string(self) -> str:
        return json.dumps(self.memory)


def hybrid_retrieve(query_text: str, time_filter: Optional[str] = None, limit: int = 3, retry: int = 0) -> List[Dict]:
    try:
        # FIXED: Use table.to_pandas() to get all data, then filter manually
        # This avoids LanceDB's full-text search requirement
        all_data = table.to_pandas()
        
        if all_data.empty:
            return []
        
        # Manual filtering: search for query_text in log_message
        query_lower = query_text.lower()
        mask = all_data['log_message'].str.lower().str.contains(query_lower, na=False)
        
        if time_filter:
            mask = mask & all_data['timestamp'].str.startswith(time_filter)
        
        filtered_results = all_data[mask].head(limit).to_dict('records')
        
        # Retry if no results
        if not filtered_results and retry < 2:
            print(f"[*] No results on attempt {retry + 1}, retrying in 2 seconds...")
            time.sleep(2)
            return hybrid_retrieve(query_text, time_filter, limit, retry + 1)
        
        if not filtered_results:
            return []
        
        return [
            {
                "timestamp": r.get("timestamp", ""),
                "radar_id": r.get("radar_id", ""),
                "azimuth": r.get("azimuth", 0),
                "elevation": r.get("elevation", 0),
                "voltage": r.get("voltage_mv", 0),
                "error_code": r.get("error_code", ""),
                "message": r.get("log_message", "")
            } for r in filtered_results
        ]
    except Exception as e:
        print(f"[ERROR] hybrid_retrieve failed: {str(e)}")
        # Fallback: return first 3 records
        try:
            all_data = table.to_pandas()
            results = all_data.head(3).to_dict('records')
            return [
                {
                    "timestamp": r.get("timestamp", ""),
                    "radar_id": r.get("radar_id", ""),
                    "azimuth": r.get("azimuth", 0),
                    "elevation": r.get("elevation", 0),
                    "voltage": r.get("voltage_mv", 0),
                    "error_code": r.get("error_code", ""),
                    "message": r.get("log_message", "")
                } for r in results
            ]
        except:
            return []


def process_radar_inquiry(user_question: str, memory_engine: ContinuumMemory) -> str:
    try:
        print(f"\n[User Inquiry]: {user_question}")

        retrieved_logs = hybrid_retrieve(user_question)
        
        if not retrieved_logs:
            print("[*] Using fallback: returning all logs for analysis")
            # Fallback: get all logs if search fails
            try:
                all_data = table.to_pandas()
                if not all_data.empty:
                    all_results = all_data.head(10).to_dict('records')
                    retrieved_logs = [
                        {
                            "timestamp": r.get("timestamp", ""),
                            "radar_id": r.get("radar_id", ""),
                            "azimuth": r.get("azimuth", 0),
                            "elevation": r.get("elevation", 0),
                            "voltage": r.get("voltage_mv", 0),
                            "error_code": r.get("error_code", ""),
                            "message": r.get("log_message", "")
                        } for r in all_results
                    ]
                else:
                    return "[ERROR] No logs available in database."
            except Exception as e:
                return f"[ERROR] Fallback retrieval failed: {str(e)}"

        evaluation_prompt = f"""
You are an automated Radar Telemetry Critic. Analyze the user inquiry and the initial retrieved data.

User Query: {user_question}
Initial Results: {json.dumps(retrieved_logs)}

Respond in JSON format:
{{"needs_more_data": true/false, "refined_search_query": ""}}
"""

        eval_output = call_local_qwen(evaluation_prompt, json_mode=True)
        
        try:
            eval_json = json.loads(eval_output)
        except json.JSONDecodeError:
            eval_json = {"needs_more_data": False, "refined_search_query": ""}

        if eval_json.get("needs_more_data") and eval_json.get("refined_search_query"):
            refined_query = eval_json['refined_search_query']
            print(f"[*] Refined Query: '{refined_query}'")
            secondary_logs = hybrid_retrieve(refined_query, limit=2)
            retrieved_logs.extend([log for log in secondary_logs if log not in retrieved_logs])

        final_synthesis_prompt = f"""
You are an expert military radar diagnostics model. Synthesize a technical report.

[Memory]: {memory_engine.get_context_string()}
[Logs]: {json.dumps(retrieved_logs, indent=2)}
[Request]: {user_question}

Provide a clear technical summary.
"""

        response = call_local_qwen(final_synthesis_prompt)
        
        if response.startswith("Error"):
            return f"[ERROR] LLM failed: {response}"

        for log in retrieved_logs:
            error_code = log.get("error_code", "")
            radar_id = log.get("radar_id", "")
            if error_code and error_code != "OK" and radar_id:
                memory_engine.update_state(error_code, radar_id)

        return response
        
    except Exception as e:
        return f"[ERROR] process_radar_inquiry failed: {str(e)}"


if __name__ == "__main__":
    print("[1] Seeding Sample Radar Telemetry Records into LanceDB...")

    sample_telemetry = [
        {
            "id": "1", "timestamp": "2026-05-22 14:00:00", "radar_id": "AN-FPS-117_A",
            "azimuth": 45.2, "elevation": 12.1, "voltage_mv": 5000, "error_code": "OK",
            "log_message": "Normal sweep operations. Thermal gradients stable within threshold limits."
        },
        {
            "id": "2", "timestamp": "2026-05-22 14:02:15", "radar_id": "AN-FPS-117_A",
            "azimuth": 89.7, "elevation": 11.5, "voltage_mv": 3100, "error_code": "ERR_403_UNDERVOLT",
            "log_message": "Critical Alert: Shifter power rail bus voltage dropped below safe margins. Motor torque stuttering."
        },
        {
            "id": "3", "timestamp": "2026-05-22 14:05:00", "radar_id": "AN-FPS-117_A",
            "azimuth": 120.4, "elevation": -2.0, "voltage_mv": 4950, "error_code": "ERR_501_ALIGN_FAIL",
            "log_message": "Mechanical block or tracking lock failure. Beam array safe mode engaged due to primary drive misalignment."
        }
    ]

    try:
        table.add(sample_telemetry)
        print("[✔] Ingestion complete.")
        print("[*] Waiting for embedding index to build...")
        time.sleep(2)
    except Exception as e:
        print(f"[!] Ingestion skipped: {type(e).__name__}")

    radar_memory = ContinuumMemory()

    inquiry_1 = "Explain why the radar unit array suddenly switched into safe mode."
    report_1 = process_radar_inquiry(inquiry_1, radar_memory)
    print("\n--- FINAL TECHNICAL SUMMARY REPORT ---")
    print(report_1)

    inquiry_2 = "What active anomalies are recorded in our state machine right now?"
    print(f"\n[User Inquiry]: {inquiry_2}")
    print(f"[Current Memory Output]:\n{json.dumps(radar_memory.memory, indent=2)}")

/var/folders/x1/wk2m5xyj59b6qqyrrvjv1bhh0000gn/T/ipykernel_13212/1862325930.py:68: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name in db.table_names():


[1] Seeding Sample Radar Telemetry Records into LanceDB...
[✔] Ingestion complete.
[*] Waiting for embedding index to build...

[User Inquiry]: Explain why the radar unit array suddenly switched into safe mode.
[*] No results on attempt 1, retrying in 2 seconds...
[*] No results on attempt 2, retrying in 2 seconds...
[*] Using fallback: returning all logs for analysis

--- FINAL TECHNICAL SUMMARY REPORT ---
### Technical Report: Radar Unit Array Safe Mode Activation

#### Summary:
The radar unit array, identified as **AN-FPS-117_A**, has been observed to switch into safe mode on multiple occasions. This report synthesizes the recent logs and known critical faults to provide a technical explanation for these occurrences.

#### Key Observations:

1. **Critical Faults:**
   - **ERR_403_UNDERVOLT:** Indicates that the shifter power rail bus voltage has dropped below safe margins, leading to motor torque stuttering.
   - **ERR_501_ALIGN_FAIL:** Signifies a mechanical block or tracking lock 